In [6]:
import pandas as pd
import numpy as np
from skbio.diversity import beta_diversity

In [19]:
abundance = pd.read_csv('../goll_et.al_2020_IBS_FMT/processed/abundance_normalized_table.csv')
rel_abundance = abundance.copy()
column_sums = rel_abundance.drop('taxon_id', axis=1).sum(axis=0)
rel_abundance.iloc[:, 1:] = rel_abundance.iloc[:, 1:].div(column_sums, axis=1)
# compute sample-level bray-curtis distance
rel_abundance_T = rel_abundance.set_index('taxon_id').transpose()
rel_abundance_T.index.name = 'sample_id'
bray_curtis_distance = beta_diversity(
    metric='braycurtis',
    counts=rel_abundance_T.values,
    ids=rel_abundance_T.index
)
bray_curtis_distance = pd.DataFrame(
    bray_curtis_distance.data,
    index=bray_curtis_distance.ids,
    columns=bray_curtis_distance.ids
)
bray_curtis_distance.to_csv('./data/bray_curtis_distance.csv', index=True)
DonorRecipientMapping = pd.read_csv('../goll_et.al_2020_IBS_FMT/raw/DonorRecipientMapping.csv')[['recipient', 'clinical_response']]
recipient = (
    DonorRecipientMapping
    .assign(
        recipient_t0=lambda x: x['recipient'].str.split('_').str[0],
        recipient_t6=lambda x: x['recipient'].str.split('_').str[0].str.replace('-0', '-6')
    )
    .drop(columns=['recipient'])
    .melt(id_vars=['clinical_response'], var_name='recipient', value_name='sample_id')
    .drop(columns=['recipient'])
)
taxonomy = pd.read_csv('../goll_et.al_2020_IBS_FMT/processed/full_taxonomy.csv')

rel_abundance_phylum = (
    rel_abundance
    .merge(taxonomy, on='taxon_id', how='left')
    .drop(columns=['strain'])
    .melt(id_vars=['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'], var_name='sample_id', value_name='abundance')
    .groupby(['phylum', 'sample_id'], as_index=False)['abundance']
    .sum()
    .pivot_table(index='phylum', columns='sample_id', values='abundance', fill_value=0)
    .transpose()
    .merge(recipient, on='sample_id', how='left')
    .fillna('donor')
    .assign(time=lambda x: np.where(
        x['sample_id'].str.contains('-6'),
        'T6',
        'T0'
    ))
)
rel_abundance_phylum['time'] = rel_abundance_phylum.apply(lambda x: 'donor' if x['sample_id'].startswith('D') else x['time'], axis=1)
rel_abundance_phylum[['sample_id', 'Bacteroidetes', 'clinical_response', 'time']].to_csv('./data/supp_2A.csv')
rel_abundance_phylum.to_csv('./data/supp_2B.csv')
rel_abundance_phylum

,sample_id,Actinobacteria,Bacteroidetes,Crenarchaeota,Cyanobacteria,Euryarchaeota,Firmicutes,Fusobacteria,Planctomycetes,Proteobacteria,Spirochaetes,Synergistetes,Tenericutes,Thaumarchaeota,Verrucomicrobia,clinical_response,time
0,11-0,0.088403,0.100094,0.000000e+00,1.511947e-05,0.000016,0.563546,0.000033,6.165993e-08,0.000920,4.334066e-06,0.000179,1.147067e-05,0.000000e+00,0.246778,responder,T0
1,11-6,0.016546,0.716328,6.809451e-08,2.845734e-04,0.018757,0.089788,0.000014,4.716928e-08,0.059768,8.104606e-07,0.000032,1.554911e-06,2.157004e-07,0.098479,responder,T6
2,12-0,0.134220,0.413521,0.000000e+00,3.168257e-04,0.000009,0.435591,0.000128,8.489238e-07,0.006784,5.525065e-06,0.000036,1.522121e-05,0.000000e+00,0.009374,non-responder,T0
3,12-6,0.735415,0.082264,0.000000e+00,1.573480e-05,0.000006,0.166974,0.000021,4.251221e-07,0.002717,1.953056e-06,0.000064,1.346310e-06,3.049476e-07,0.012520,non-responder,T6
4,16-0,0.070449,0.086953,0.000000e+00,3.118708e-05,0.000018,0.674787,0.000065,1.982614e-07,0.000616,8.671142e-06,0.000132,1.564515e-05,0.000000e+00,0.166923,responder,T0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,D-6Fresk,0.033368,0.808858,0.000000e+00,4.459142e-07,0.000002,0.149788,0.000016,0.000000e+00,0.007952,3.010951e-06,0.000006,1.251300e-06,0.000000e+00,0.000003,donor,donor
64,D-6Fryst,0.002554,0.851479,0.000000e+00,2.551071e-06,0.000001,0.140569,0.000008,2.756989e-07,0.005364,1.507242e-06,0.000018,8.948349e-07,0.000000e+00,0.000002,donor,donor
65,D-7Fryst,0.002700,0.877572,0.000000e+00,1.204752e-06,0.000002,0.114506,0.000012,2.231996e-07,0.005197,1.510760e-06,0.000005,0.000000e+00,0.000000e+00,0.000003,donor,donor
66,D-9Feresk,0.001168,0.920071,0.000000e+00,3.693247e-07,0.000002,0.068858,0.000007,1.596544e-07,0.009889,2.493795e-07,0.000003,0.000000e+00,0.000000e+00,0.000002,donor,donor
